In [ ]:
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [ ]:
# run in parallel

import multiprocessing as mp
from pathlib import Path

from general_utils import find_ephys_sessions
from monotonic_mp import run_all_sessions_parallel

# IMPORTANT in notebooks
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")
OUT_DIR = Path("/root/capsule/scratch/monotonic_outputs_n_4")

LATENT_NAMES = [
    "ForagingCompareThreshold-value-1",
    "QLearning_L2F1_softmax-deltaQ-1",
    "QLearning_L2F1_softmax-sumQ-1",
]

ACTIVITY_WINDOWS = [(-1, 0), (0.2, 2),(-3,0)]

_, _, sessions = find_ephys_sessions()


_results = run_all_sessions_parallel(
    list(sessions),
    psth_dir=PSTH_DIR,
    results_dir=RESULTS_DIR,
    out_dir=OUT_DIR,
    latent_names=LATENT_NAMES,
    activity_windows=ACTIVITY_WINDOWS,
    max_workers=4,   # scale up later if stable
)


failed = [r for r in _results if not r.get("ok", False)]

print(f"Failed sessions: {len(failed)}")

failed_sessions = [r.get("session") for r in failed]
print(failed_sessions[:20])

sessions=failed_sessions
_results = run_all_sessions_parallel(
    list(sessions),
    psth_dir=PSTH_DIR,
    results_dir=RESULTS_DIR,
    out_dir=OUT_DIR,
    latent_names=LATENT_NAMES,
    activity_windows=ACTIVITY_WINDOWS,
    max_workers=4,   # scale up later if stable
)

In [ ]:
# -----------------------------
# Load summary (as you already do)
# -----------------------------
import importlib
import behavior_qc_visualization

importlib.reload(behavior_qc_visualization)


from behavior_qc_visualization import load_behavior_model_summary_csv
from behavior_qc_metrics_summary import append_model_criteria_result

summary = load_behavior_model_summary_csv(
    "/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv"
    # "/root/capsule/scratch/behavior_model_summary_general_behavior.csv"
)
summary = append_model_criteria_result(summary)


# -----------------------------
# Load combined_df + summarize (UPDATED: add summary + criteria_col)
# -----------------------------
from check_monotonic import load_and_combine_monotonic_unit_dfs
from check_monotonic import summarize_significant_and_monotonic_fractions

combined_df = load_and_combine_monotonic_unit_dfs(
    "/root/capsule/scratch/monotonic_outputs_n_4/ForagingCompareThreshold-value-1/win_m1__0",
    pattern="*.csv",
    recursive=True,
    add_source_file=True,
)
print("combined_df shape (before filtering inside function):", combined_df.shape)

out = summarize_significant_and_monotonic_fractions(
    combined_df,
    alpha=0.05,
    p_col="spearman_p",
    monotonic_col="is_monotonic_gt_thr",
    include_overall=True,
    summary=summary,  # NEW
    # Default criteria_col is 'QLearning_L1F1_CK1_softmax_pass_all_criteria'
    # Override here if you want a different model's criteria:
    # criteria_col="QLearning_L2F1_softmax_pass_all_criteria",
)

# Optional: see how many rows survived the session filter
if out.get("session_filter_applied", False):
    print(
        "Session filter applied:",
        out["criteria_col"],
        "| rows:",
        out["n_rows_before_session_filter"],
        "->",
        out["n_rows_after_session_filter"],
        "| n_sessions_passing:",
        out["n_sessions_passing_criteria"],
    )

overall = out["overall"]
print("Fraction significant (among valid p):", overall["frac_significant_among_valid_p"])
print("Fraction monotonic among significant:", overall["frac_monotonic_among_significant"])
print("n_sig:", overall["n_significant"], "n_sig_monot:", overall["n_significant_and_monotonic"])

sig_df = overall["significant_df"]
sig_monot_df = overall["significant_monotonic_df"]

for label, res in out["by_brain_region_group"].items():
    print(
        label,
        "n_total:", res["n_total_rows"],
        "frac_sig:", res["frac_significant_among_valid_p"],
        "frac_monot|sig:", res["frac_monotonic_among_significant"],
        "n_sig:", res["n_significant"],
        "n_sig_monot:", res["n_significant_and_monotonic"],
    )


In [ ]:
combined_df['brain_region'].dropna().unique()


In [ ]:
out['by_brain_region_group'].keys()

In [ ]:
out['by_brain_region_group']['[PL5,PL6a,ILA5,ILA6a]']['significant_monotonic_df']

In [ ]:
df = out['by_brain_region_group']['[SI,MA]']['significant_monotonic_df']

row = df.loc[
    (df['session_name'] == 'ecephys_776293_2025-02-18_12-51-36_sorted_2025-03-30_05-50-33') &
    (df['unit_index'] == 1796)
]

row


In [ ]:
row['latent_name']

In [ ]:
print(row['q0_mean_activity'])
print(row['q1_mean_activity'])
print(row['q2_mean_activity'])
print(row['q3_mean_activity'])


print(row['q0_trial_mean_activity_list'])
print(row['q1_trial_mean_activity_list'])
print(row['q2_trial_mean_activity_list'])
print(row['q3_trial_mean_activity_list'])

In [ ]:
row['latent_name']

In [ ]:
row = out['by_brain_region_group']['[SI,MA]']['significant_monotonic_df'].loc[
    (df['session_name'] == 'ecephys_776293_2025-02-18_12-51-36_sorted_2025-03-30_05-50-33') &
    (df['unit_index'] == 1796)
].iloc[0]

row[['activity_window_start','activity_window_end','n_bins_effective',
     'q0_n','q1_n','q2_n','q3_n',
     'q0_mean_activity','q1_mean_activity','q2_mean_activity','q3_mean_activity',
     'is_monotonic','monotonic_direction','monotonic_diffs',
     'is_monotonic_gt_thr','monotonic_direction_gt_thr','monotonic_diffs_gt_thr',
     'n_trials_used','n_trials_used_gt_thr']]


In [ ]:
out['by_brain_region_group']['[SI,MA]' ]

In [ ]:
df = out['by_brain_region_group']['[MD]']['significant_monotonic_df']

row_1976 = df[df['unit_index'] == 1976]

row_1976


In [ ]:
row = combined_df[
    (combined_df['unit_index'] == 365) &
    (combined_df['brain_region'].isin(['PL5','PL6a','ILA5','ILA6a']))
]

row


In [ ]:
# Example: randomly open 8 significant & monotonic neurons

import importlib
import check_monotonic

# Reload the entire module
importlib.reload(check_monotonic)

from check_monotonic import show_unit_figures_from_df_inline

opened_paths = show_unit_figures_from_df_inline(
    out['by_brain_region_group']['[SI,MA]' ]['significant_monotonic_df'],   # your filtered DataFrame
    root="/root/capsule/scratch/raster_plot",
    session_col="session_name",
    latent_col="latent_name",
    unit_col="unit_index",
    n=100,                        # randomly open 8 figures
    random_state=41,            # reproducible sampling
)

print(f"Opened {len(opened_paths)} figures.")


In [ ]:
import importlib
import plot_raster

importlib.reload(plot_raster)

from plot_raster import plot_raster_and_quantile_psth_by_latent



from pathlib import Path
import numpy as np

from general_utils import find_ephys_sessions, smart_read_csv
from behavior_utils import find_trials, get_fitted_model_names
from nwb_utils import NWBUtils
from create_psth import load_zarr
from plot_raster import plot_raster_and_quantile_psth_by_latent


PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")
OUTDIR = Path("/root/capsule/scratch/raster_plot")

LATENTS = [
    "QLearning_L2F1_softmax-deltaQ-1",
    "QLearning_L2F1_softmax-reward",
    "QLearning_L2F1_softmax-sumQ-1",
]
latent_names = ['deltaQ-1','reward','sumQ-1']

ALIGN_TO = "go_cue"
TIME_WIN = (-3, 4)


OUTDIR.mkdir(parents=True, exist_ok=True)
_, _, sessions = find_ephys_sessions()
print("Sessions:", sessions)

sessions=["ecephys_769884_2025-01-15_16-12-59_sorted_2025-04-24_17-11-57"]




for s in sessions:
    print(f"\n[{s}] models:", get_fitted_model_names(session_name=s))
    nwb, _ = NWBUtils.combine_nwb(session_name=s)
    psth = load_zarr(str(PSTH_DIR / f"{s}_0.2s.zarr"))
    beh = smart_read_csv(str(RESULTS_DIR / f"behavior_summary-{s}.csv"))
    trials = np.asarray(find_trials(nwb, "response"))
    save_dir = OUTDIR / s
    save_dir.mkdir(exist_ok=True)

    i=0
    for col in LATENTS:
        if col not in beh.columns:
            print(f"  skip: {col} (missing)")
            continue
        vals = beh[col][0]

        plot_raster_and_quantile_psth_by_latent(
            source=psth,
            latent_values=vals,              # full vector
            latent_trial_ids=trials,
            unit_ids=[365],
            align_to_event=ALIGN_TO,
            time_window=TIME_WIN,
            n_bins=4,
            binning="equal",
            quantile_stat="mean",
            ci="sem",
            title_prefix=None,
            figsize=(6, 5),
            save_path="/root/capsule/scratch/tmp",
            cmap_name="coolwarm",
            raster_colormap=False,
            show_colormap=True,
            save_prefix=f'{col}_',
            latent_name=latent_names[i],
            show=True,
            min_trial_rate=0,
            sort_order='not_sort',
            dpi=600
        )
        i=i+1
        print(f"  plotted: {col}")






In [ ]:
/root/capsule/scratch/raster_plot/ecephys_776293_2025-02-18_12-51-36_sorted_2025-03-30_05-50-33/QLearning_L2F1_softmax-deltaQ-1/QLearning_L2F1_softmax-deltaQ-1_unit_1796.png
/root/capsule/scratch/raster_plot/ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48/ForagingCompareThreshold-value-1/ForagingCompareThreshold-value-1_unit_914.png
/root/capsule/scratch/raster_plot/ecephys_776293_2025-02-18_12-51-36_sorted_2025-03-30_05-50-33/ForagingCompareThreshold-value-1/ForagingCompareThreshold-value-1_unit_1976.png
/root/capsule/scratch/raster_plot/ecephys_769884_2025-01-15_16-12-59_sorted_2025-04-24_17-11-57/ForagingCompareThreshold-value-1/ForagingCompareThreshold-value-1_unit_365.png